# Lab 4, part 2: Naive Bayes

In part 1 you computed `P(survived | female)` by keeping the rows where `Sex` was female and taking a proportion. We will do something similar here, but at a larger scale, and implement the Naive Bayes algorithm.

## Part 0: The data

`sms.csv` holds 5,574 real text messages, each labelled `ham` (ordinary) or `spam`. It was collected for research in 2011 and has been used for this kind of classifier.

In [ ]:
# PROVIDED
import math
import random
import re

import pandas as pd

sms = pd.read_csv("sms.csv")
print(f"{len(sms)} messages")
print(sms.label.value_counts().to_string())
sms.head()

### Baseline Accuracy


Accuracy is the number of correct classifications divided by the total number of trials.

Note that 4,827 of the 5,574 messages are ham. Imagine a classifier that guesses at random, and another that always guesses ham. What are their accuracies?

In [ ]:
# PROVIDED
print(f"always guessing 'ham' is right {(sms.label == 'ham').mean():.1%} of the time")

### Turning a message into words

A classifier cannot work with a string, so the first step is to chop it into pieces. The provided code below will lowercase everything, keep runs of letters and apostrophes, throw the rest away.

In [ ]:
# PROVIDED
def words(text):
    """Split a message into lowercase words."""
    return re.findall(r"[a-z']+", str(text).lower())


print(words("Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005"))

## Part 1: Train/Test split

When training a machine learning algorithm, you should not evaluate the model on the same data it is trained with. The code below will split the dataset randomly into a train and test set with an 80/20 split.

In [ ]:
# PROVIDED
order = list(range(len(sms)))
random.Random(0).shuffle(order)
cut = int(0.8 * len(order))

train = sms.iloc[order[:cut]].reset_index(drop=True)
test = sms.iloc[order[cut:]].reset_index(drop=True)

print(f"train: {len(train)} messages, {(train.label == 'spam').mean():.1%} spam")
print(f"test:  {len(test)} messages, {(test.label == 'spam').mean():.1%} spam")

## Part 2: Training by counting

The model needs two things, and both are proportions of the kind you computed in part 1.

**The priors:** `P(spam)` and `P(ham)`. This is computed by marginalizing.

**The word probabilities:** For every word, `P(word | spam)` and `P(word | ham)`. Keep only the spam messages, count how often the word appears among all the words in them, and divide. This is a conditional.

$$P(w \mid c) = \frac{\text{times } w \text{ appears in class } c}{\text{total words in class } c}$$

### Exercise 1: build the model

Compute and keep:

* `priors`: a dictionary mapping `"ham"` and `"spam"` to prior probs
* `counts`: for each class, how many times each word appears across all training messages of that class
* `totals`: for each class, the total number of words across all its training messages
* `vocabulary`: the set of every word appearing anywhere in training

Then print the vocabulary size and the five most common words in each class.

In [ ]:
# Your code here.

In [ ]:
# PROVIDED: prints out your stuff
print(f"P(ham)  = {priors['ham']:.4f}")
print(f"P(spam) = {priors['spam']:.4f}")
print(f"vocabulary: {len(vocabulary)} words")
print(f"total words: {totals['ham']} in ham, {totals['spam']} in spam")
for label in ("ham", "spam"):
    top = sorted(counts[label].items(), key=lambda pair: -pair[1])[:5]
    print(f"  most common in {label}: {[word for word, n in top]}")

<details><summary>Hint 1: the shape</summary>

One pass over the training messages. For each message, split it into words and, for every word, add one to that word's tally for that message's class and one to the class total.

Nested dictionaries work well: `counts["spam"]["free"]` is how many times "free" appears across all spam.

</details>

<details><summary>Hint 2: accumulating</summary>

`counts[label].get(word, 0)` will get a (current) count of the word, and return 0 if it is not found.

`counts[label][word] = counts[label].get(word, 0) + 1` may be a convient line...

</details>

<details><summary>Hint 3: the answers</summary>

`P(ham) = 0.8686`, `P(spam) = 0.1314`. (Note these are the proportions in the *training* half, not the whole corpus.)

The vocabulary has **7,070** words. There are 54,691 words in ham and 13,391 in spam.

Most common in ham: `['i', 'you', 'to', 'the', 'a']`. Most common in spam: `['to', 'call', 'a', 'you', 'your']`.

</details>

## Part 3: Scoring a message

To classify a message, compute (estimate) the probability of each class and choose the more likely one.

The score for class $c$ is the probability of seeing that class *and* those words:

$$P(c) \times \prod_{w \in \text{message}} P(w \mid c)$$

The product is where the word "naive" comes from and assumes independence between words. This is probably not true, but let's assume that for now.

### Exercise 2: score, the obvious way

Write `score(text, label)` returning the above product. Skip any word that is not in the vocabulary.

Then classify every message in the test set and report the accuracy.

In [ ]:
# Your code here.

<details><summary>Hint 1: the loop</summary>

Start a running total at `priors[label]`, then multiply by `counts[label][word] / totals[label]` for each word in the message. Use `.get(word, 0)` so a word absent from that class does not raise.

</details>

<details><summary>Hint 2: the answers</summary>

Accuracy **0.9561**, which is 1,066 of 1,115. It predicts spam 118 times and ham 997 times, while the test set holds 161 spam and 954 ham.

</details>

### What went wrong

95.6% sounds respectable, but remember the baseline. Guessing "ham" every time scores 86.6%, so the model has bought you nine percentage points.

Look at the individual scores.

In [ ]:
# PROVIDED
try:
    shown = 0
    for text, actual in zip(test.text, test.label):
        ham_score, spam_score = score(text, "ham"), score(text, "spam")
        if ham_score == 0.0 or spam_score == 0.0:
            print(f"  ham={ham_score:<12.4g} spam={spam_score:<12.4g} actual={actual}")
            print(f"    {text[:70]}")
            shown += 1
        if shown == 4:
            break

    both_zero = 0
    for text in test.text:
        if score(text, "ham") == 0.0 and score(text, "spam") == 0.0:
            both_zero += 1
    print(f"\n{both_zero} of {len(test)} test messages score exactly 0.0 under "
          f"*both* classes.")
    print("With nothing to compare, those fall through to whichever branch the "
          "tie-break happens to take.")
except NameError:
    print("⏭  Finish Exercise 2 first.")

The culprit is a single zero. If one word of a message never appeared in spam during training, `P(word | spam)` is 0, and multiplying by it wipes out the entire product no matter how spammy everything else was. One unseen word outvotes twenty damning ones.

This is bad modelling: a count of zero in a sample of 15,000 words is not good evidence that something is *impossible*.

### The fix: pretend you saw everything once

Add one to every count before dividing. Every word in the vocabulary, including ones never seen in this class, gets a small but nonzero probability:

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{total}(c) + |V|}$$

The $|V|$ on the bottom — the vocabulary size — is there so the probabilities still sum to 1 once you have added $|V|$ fake observations. This is **Laplace smoothing**, and the one is arbitrary (the number you add is called the smoothing parameter).

### Exercise 3: smooth it

Write `smoothed_score(text, label)` using the formula above, classify the test set again, and report the accuracy.

In [ ]:
# Your code here.

<details><summary>Hint 1: two changes</summary>

The numerator gains `+ 1` and the denominator gains `+ len(vocabulary)`. Nothing else moves.

</details>

<details><summary>Hint 2: the answers</summary>

Accuracy **0.9839**, which is 1,097 of 1,115, predicting spam 159 times.

</details>

## What does 98.4% mean?

Accuracy counts how often the classifier is right. On data that is 86.6% one class, that is a low bar, and it hides the question you actually care about: of the spam, how much did it catch, and of the ordinary messages, how many did it throw away?

The table below splits the 1,115 test messages four ways.

In [1]:
# PROVIDED
try:
    predictions = [smoothed_classify(text) for text in test.text]
    table = pd.crosstab(pd.Series(list(test.label), name="actually"),
                        pd.Series(predictions, name="predicted"))
    display(table)

    caught = table.loc["spam", "spam"]
    missed = table.loc["spam", "ham"]
    flagged = table.loc["ham", "spam"]
    print(f"spam caught          : {caught} of {caught + missed}  "
          f"(recall {caught / (caught + missed):.3f})")
    print(f"ham wrongly flagged  : {flagged} of {flagged + table.loc['ham', 'ham']}")
    print(f"of messages it called spam, this share really were: "
          f"{caught / (caught + flagged):.3f}  (precision)")
except NameError:
    print("⏭  Finish Exercise 3 first.")

⏭  Finish Exercise 3 first.


Those two mistakes are not equally bad, and which one you would rather make is not a question statistics can answer for you. A spam message in the inbox is an annoyance. A real message in the spam folder might be a job offer.

Accuracy adds the two together and reports a single number, which means it has quietly made that decision on your behalf.